In [2]:
import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

words = open('names.txt', 'r').read().splitlines()
print(words[:5])
print(len(words))

cuda
['emma', 'olivia', 'ava', 'isabella', 'sophia']
32033


In [3]:
# encode chars to integers
chars = sorted(list(set(''.join(words))))
stoi = { ch:i+1 for i,ch in enumerate(chars)}
stoi['.'] = 0
itos = { i:ch for ch,i in stoi.items()}
vocab_size = len(itos)
print(itos)

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello"))
print(decode([8, 9, 0]))
print(encode(words[0][0]))

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
[8, 5, 12, 12, 15]
hi.
[5]


In [4]:
# build dataset
'''
[...] -> [b]
[..b] -> [o]
[.bo] -> [b]
[bob] -> [.]
'''
block_size = 4

def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        # sliding window
        for ch in w + '.':
            ix = stoi[ch] # encode(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

n1 = int(0.9*len(words))

X, Y = build_dataset(words)
Xtr, Ytr = build_dataset(words[:n1])
Xval, Yval = build_dataset(words[n1:])

print(X[:5])
print(Y[:5])
len(Xtr)

tensor([[ 0,  0,  0,  0],
        [ 0,  0,  0,  5],
        [ 0,  0,  5, 13],
        [ 0,  5, 13, 13],
        [ 5, 13, 13,  1]])
tensor([ 5, 13, 13,  1,  0])


205411

In [5]:
batch_size = 32

def get_batch(split):
    dataX = Xtr if split == 'train' else Xval
    dataY = Ytr if split == 'train' else Yval

    ix = torch.randint(low=0, high=dataX.shape[0], size=(batch_size, ))
    x = torch.stack([dataX[i] for i in ix])
    y = torch.stack([dataY[i] for i in ix])

    x, y = x.to(device), y.to(device)
    return x, y

# Xb, Yb = get_batch('train')
# print(decode(Xb[:1][0].tolist()))
# print(decode(Yb[:1].tolist()))


In [6]:
# create embedding -> FFN (tanh non-linearity)
emb_size = 64
n_hidden = 64

C = torch.randn((vocab_size, emb_size), device=device)
P = torch.randn((block_size, emb_size), device=device)
W1 = torch.randn((block_size * emb_size, n_hidden), device=device)
B1 = torch.randn((n_hidden), device=device)
W2 = torch.randn((n_hidden, vocab_size), device=device)
B2 = torch.randn((vocab_size), device=device)
parameters = [C, P, W1, B1, W2, B2]
for p in parameters:
    p.requires_grad = True


In [18]:
max_iters = 10000
lr = 3e-3

for i in range(max_iters):
    Xb, Yb = get_batch('train') # (B, T) -> (batch_size, block_size)

    emb = C[Xb] # (B, T, C) -> (batch_size, block_size, emb_size)
    pos = P # (T, C) -> (1, block_size, emb_size)

    # concat (B, T*C) -> tanh linear
    emb += pos
    x = emb.view(emb.shape[0], -1)
    x = torch.tanh(x @ W1 + B1)
    x = x @ W2 + B2

    # cross entropy loss (negative log likelihood)
    loss = F.cross_entropy(x, Yb)

    # zero grad
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -lr * p.grad

    if i % 1000 == 0:
        print(loss.item())

2.566509246826172
2.986865282058716
3.775552272796631
2.7894482612609863
2.785863161087036
2.3739418983459473
2.596895694732666
3.114583969116211
2.935171604156494
2.6534430980682373


In [ ]:
# AdamW update variant
optimizer = torch.optim.AdamW(parameters, lr=lr)
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()

In [19]:
torch.save(parameters, 'parameters.pt')
parameters = torch.load('parameters.pt', weights_only=True)
C, P, W1, B1, W2, B2 = parameters

for p in parameters:
    p.requires_grad = True

In [21]:
# generation
@torch.no_grad()
def generate():
    out = []
    context = [0] * block_size

    out = encode("eil")
    context = encode(".eil") 
    while True:
        Xgen = torch.tensor([context], device=device)
        emb = C[Xgen] # (B, T, C) -> (batch_size, block_size, emb_size)
        emb += P
        # concat (B, T*C) -> tanh linear
        x = emb.view(emb.shape[0], -1)
        x = torch.tanh(x @ W1 + B1)
        x = x @ W2 + B2
        probs = F.softmax(x, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(out)
    print(decode(out))

for _ in range(5):
    generate()

    


[5, 9, 12, 9, 5, 0]
eilie.
[5, 9, 12, 1, 1, 0]
eilaa.
[5, 9, 12, 1, 0]
eila.
[5, 9, 12, 9, 14, 0]
eilin.
[5, 9, 12, 1, 0]
eila.
